# CO₂ Storage Capacity Assessment

This notebook runs a probabilistic static storage-capacity assessment using:

$$SC = GRV \times (N/G) \times \phi \times \rho_{CO_2} \times S_{eff}$$

Change the values in the **Editable inputs** cell, then choose **Runtime → Run all**. The example values represent the Rødby Bunter Sandstone assessment.

In [ ]:
# Install the latest package and plotting tools from GitHub.
%pip install -q "git+https://github.com/AnaSoles/ggg-co2-storage-eval.git" matplotlib pandas

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from storageeval import Distribution, StorageSite, simulate

plt.style.use("seaborn-v0_8-whitegrid")

## Editable inputs

Enter minimum, most likely, and maximum values. Fractions such as porosity must be decimals: `0.23` means 23%. The Rødby values come from GEUS Report 2024/18, Table 8.4.1.

**Source correction:** Table 8.4.1 prints the maximum CO₂ density as 764.0 kg/m³, but Section 8.2.4 says the maximum is 10% above the 603.6 kg/m³ mode. The internally consistent value, 663.96 kg/m³, is used below because it reproduces the report's capacity results.

In [ ]:
site_name = "Rødby – Bunter Sandstone"
iterations = 100_000
random_seed = 42

#                       minimum, most likely, maximum
grv_km3 =               (22.57, 28.21, 33.85)
net_to_gross =          (0.20,  0.25,  0.30)
porosity =              (0.184, 0.23,  0.276)
co2_density_kg_m3 =     (573.4, 603.6, 663.96)  # Table prints 764.0; Section 8.2.4 implies 663.96
storage_efficiency =    (0.05,  0.10,  0.20)

## Input parameter table

This table is generated from the editable values above, so it updates automatically when you change an input.

In [ ]:
input_table = pd.DataFrame([
    ["Gross rock volume (GRV)", "km³", "PERT", *grv_km3, "GEUS Table 8.4.1"],
    ["Net-to-gross (N/G)", "fraction", "PERT", *net_to_gross, "GEUS Table 8.4.1"],
    ["Porosity (φ)", "fraction", "PERT", *porosity, "GEUS Table 8.4.1"],
    ["In-situ CO₂ density", "kg/m³", "PERT", *co2_density_kg_m3, "GEUS §8.2.4; max = mode +10%"],
    ["Storage efficiency", "fraction", "PERT", *storage_efficiency, "GEUS Table 8.4.1"],
], columns=["Parameter", "Unit", "Distribution", "Minimum", "Mode", "Maximum", "Source / note"])
input_table

In [ ]:
site = StorageSite(
    name=site_name,
    grv=Distribution.pert(*grv_km3),
    net_to_gross=Distribution.pert(*net_to_gross),
    porosity=Distribution.pert(*porosity),
    co2_density=Distribution.pert(*co2_density_kg_m3),
    storage_efficiency=Distribution.pert(*storage_efficiency),
)

result = simulate(site, iterations=iterations, seed=random_seed)
summary = result.summary()
report_values = {"p90_mt": 68.83, "p50_mt": 103.90, "p10_mt": 148.78, "mean_mt": 107.04}
result_rows = [("P90 (conservative)", "p90_mt"), ("P50 (median)", "p50_mt"), ("P10 (upside)", "p10_mt"), ("Mean", "mean_mt")]
comparison = pd.DataFrame({
    "Notebook (Mt CO₂)": [summary[key] for _, key in result_rows],
    "GEUS Table 8.5.1 (Mt CO₂)": [report_values[key] for _, key in result_rows],
}, index=[label for label, _ in result_rows])
comparison["Difference (Mt CO₂)"] = comparison["Notebook (Mt CO₂)"] - comparison["GEUS Table 8.5.1 (Mt CO₂)"]
comparison.round(2)

The notebook and GEUS values should be very close, but not numerically identical. Both use the same static volumetric equation and independent PERT inputs. Small differences are expected because the report does not state its number of Monte Carlo iterations, random seed, or exact software implementation of PERT.

## Input uncertainty distributions

In [ ]:
labels = {
    "grv_km3": "GRV (km³)",
    "net_to_gross": "Net-to-gross",
    "porosity": "Porosity",
    "co2_density_kg_m3": "CO₂ density (kg/m³)",
    "storage_efficiency": "Storage efficiency",
}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, values) in zip(axes.flat, result.inputs.items()):
    ax.hist(values, bins=45, color="#2a6fbb", alpha=0.82)
    ax.set_title(labels[name])
    ax.set_ylabel("Simulations")
axes.flat[-1].axis("off")
fig.suptitle(f"{site_name} – input uncertainty", fontsize=15)
fig.tight_layout()
plt.show()

## Storage-capacity probability distribution

In [ ]:
fig, ax = result.plot_pdf()
plt.show()

## Exceedance curve

P90 is the capacity that has a 90% probability of being exceeded; P10 is the upside estimate.

In [ ]:
fig, ax = result.plot_exceedance()
plt.show()

## Linear capacity confidence ranges

The colored bar summarizes conservative, central, and upside capacity ranges. These are probabilistic static capacity estimates, not booked reserves.

In [ ]:
fig, ax = result.plot_capacity_ranges()
plt.show()

## Sensitivity tornado chart

Longer bars identify the assumptions with the strongest influence on calculated capacity.

In [ ]:
fig, ax = result.plot_sensitivity()
fig.set_size_inches(10, 6)
plt.show()

## Important limitation

This is a static volumetric screening assessment. It does not yet represent pressure constraints, injectivity, plume migration, dynamic reservoir simulation, or economics.